generate data

In [1]:
import jax.numpy as jnp
import jax
from jax import random

chars = "0123456789+= "
EOS = '<|EOS|>'
ctoi = {chars[i]:i for i in range(len(chars))}
ctoi[EOS] = 13
itoc = {i:chars[i] for i in range(len(chars))}
itoc[13] = EOS

MAX_PROMPT_LEN = 17
MAX_OPERAND = 999
seed = 42
b = 10

def generate_batch(b, key):
  out = []
  loss_mask = []
  for i in range(b):
    key, subkey = random.split(key)
    operands = random.randint(subkey, (2,), 0, MAX_OPERAND)
    prompt = f"{operands[0]} + {operands[1]} = "
    prompt_size = len(prompt)
    ans = f"{operands[0] + operands[1]}"
    input = "".join([prompt, ans])
    tokenized_prompt = jnp.array([ctoi[ch] for ch in input] + [13])
    non_padded_size = tokenized_prompt.shape[0]
    padded_tokenized_input = jnp.pad(tokenized_prompt, (0, MAX_PROMPT_LEN - tokenized_prompt.shape[0]), constant_values=12)
    inds = jnp.arange(MAX_PROMPT_LEN)
    input_loss_mask = jnp.logical_or(inds >= non_padded_size, inds < prompt_size)
    out.append(padded_tokenized_input)
    loss_mask.append(input_loss_mask)

  batch = jnp.vstack(out)
  loss_mask = jnp.vstack(loss_mask)

  return batch, loss_mask

# key = random.key(42)
# key, subkey = random.split(key)

# batch, loss_mask = generate_batch(b, subkey)
# final_batch_decoded = ["".join(
#     [itoc[ch] for ch in batch[bi].tolist()]
# ) for bi in range(b)]

# for i in range(len(final_batch_decoded)):
#   print(final_batch_decoded[i])
#   print(loss_mask[i])
#   print('\n\n')



model definition

prenorm + rope + swiglu with weight tying for the embed / unembed matrix

use vanilla mha since using grouped mqa or mqa is unnecessary for the scale of this exercise and we aren't doing kv cached inference.

config:

D=256
F=4*256
L=10
N=4
=> ~10.5M params

In [7]:
from dataclasses import dataclass
from flax import nnx

@dataclass
class MoEConfig:
  experts_per_token: int
  experts: int

@dataclass
class ModelConfig:
  hidden_size: int
  ffw_size: int
  layers: int
  attn_heads: int
  vocab_size: int
  moe_config: MoEConfig | None


class RMSNorm(nnx.Module):
  def __init__(self, config):
    hidden_size = config.hidden_size
    self.gamma = nnx.Param(jnp.ones((hidden_size,)))

  def __call__(self, x):
    inv_norm = jax.lax.rsqrt(jnp.mean(x**2, axis=-1) + 1e-6)[..., None]
    return self.gamma * x * inv_norm


class RoPE(nnx.Module):
  def _create_inv_freqs(self, base, head_size):
    assert head_size % 2 == 0
    inds = jnp.arange(head_size // 2)
    return base ** (-2 * inds / head_size)

  def _create_cos_sin(self, T, inv_freqs):
    # [1, T, 1, 1]
    positions = jnp.arange(T)[None, :, None, None]
    # [1, 1, 1, H]
    dupl_inv_freqs = jnp.concat([inv_freqs, inv_freqs])[None, None, None, :]
    # [1, T, 1, H]
    freqs = positions * dupl_inv_freqs
    return jnp.cos(freqs), jnp.sin(freqs)

  def _rotate_half(self, x):
    # x[B, T, N, H]
    H = x.shape[-1]
    even = jax.lax.dynamic_slice_in_dim(x, 0, H//2, axis=-1)
    odd = jax.lax.dynamic_slice_in_dim(x, H//2, H//2, axis=-1)
    return jnp.concat([-odd, even], axis=-1)

  def __init__(self, config, base=10_000):
    """

    for each pair of features in input

    x = [[x0],
        [x1]]

    apply rotation matrix:

    R = [[cos0, -sin0]
        [sin0, cos0]]

    Rx =>

    x0' = x0*cos0 - x1*sin0
    x1' = x1*cos0 + x0*sin0

    where 0 = 2*pi*f_j*t

    and f_j = base ** (-2i/d) where i is the index of the pair

    (at i = 0, we have frequency 1 (fast recurrences))
    (at i = d/2, we have frequency base (slow long term))

    pretend top half of x are "even" indices of x
    and bottom half of x are "odd" indices of x
    this doesn't matter since the score doesn't change
    and is just a permutation in projection weights


    x_even' = x_even*cos0 - x_odd*sin0
    x_odd' = x_odd*cos0 + x_even*sin0

    x is split so that top half is even and bottom half is odd

    then this essentially becomes x = x * cos + rotate_half(x) * sin
    where rotate half takes the odd and stacks on top and negates

    note the d dimension of cos and sin must follow this half convention as well
    """
    head_size = config.hidden_size // config.attn_heads
    self.base = base
    self.inv_freqs = self._create_inv_freqs(base, head_size)

  def __call__(self, x):
    """
    pretend top half of x are "even" indices of x
    and bottom half of x are "odd" indices of x
    this doesn't matter since the score doesn't change
    and is just a permutation in projection weights
    """

    B, T, N, H = x.shape
    cos, sin = self._create_cos_sin(T, self.inv_freqs)
    return x * cos + self._rotate_half(x) * sin


class MHA(nnx.Module):
  def __init__(self, config, *, rngs):
    self.config = config
    hidden_size = config.hidden_size
    self.pre_norm = RMSNorm(config)
    self.rope = RoPE(config)
    self.qkv_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (hidden_size, 3 * hidden_size)))
    self.o_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (hidden_size, hidden_size)))

  def _sdpa(self, q, k, v):
    """
    q[B, T, N, H]
    k[B, S, N, H]
    v[B, S, N, H]
    """
    B, T, N, H = q.shape
    S = k.shape[1]
    logits = jnp.einsum('btnh,bsnh->btsn', q, k)
    scaled_logits = (H ** -0.5) * logits
    # logits[B, T, S, N]
    # mask[1, T, S, 1]
    # non mask token positions are True
    mask = jnp.tril(jnp.ones((T, S), dtype=jnp.bool))[None, :, :, None]
    # [B, T, S, N]
    masked_scaled_logits = jnp.where(mask, scaled_logits, -jnp.inf)
    weights = jax.nn.softmax(masked_scaled_logits, axis=2)
    attn_out = jnp.einsum('btsn,bsnh->btnh', weights, v)
    return attn_out

  def __call__(self, x):
    # x = [B, T, D]
    B, T, D = x.shape
    N = self.config.attn_heads
    H = D//N
    t = self.pre_norm(x)
    qkv = jnp.einsum('btd,df->btf', t, self.qkv_proj)
    # q[B, T, D]
    q = jax.lax.dynamic_slice_in_dim(qkv, 0, D, axis=-1).reshape((B, T, N, H))
    q = self.rope(q)
    k = jax.lax.dynamic_slice_in_dim(qkv, D, D, axis=-1).reshape((B, T, N, H))
    k = self.rope(k)
    v = jax.lax.dynamic_slice_in_dim(qkv, 2*D, D, axis=-1).reshape((B, T, N, H))
    # [B, T, N, H] -> [B, T, D]
    attn_out = self._sdpa(q, k, v).reshape((B, T, D))
    out = jnp.einsum('btd,df->btf', attn_out, self.o_proj)
    return x + out


# first do dense then incorporate sparse moe
class MLP(nnx.Module):
  def __init__(self, config, *, rngs):
    self.config = config
    hidden_size = config.hidden_size
    ffw_size = config.ffw_size
    moe_config = config.moe_config
    self.pre_norm = RMSNorm(config)
    if moe_config:
      E, k = moe_config.experts, moe_config.experts_per_token
      self.router = nnx.Param(0.02 * jax.random.normal(rngs.params(), (hidden_size, E)))
      self.glu_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (E, hidden_size, 2*ffw_size)))
      self.down_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (E, ffw_size, hidden_size)))
    else:
      self.glu_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (hidden_size, 2*ffw_size)))
      self.down_proj = nnx.Param(0.02 * jax.random.normal(rngs.params(), (ffw_size, hidden_size)))
    self.beta = nnx.Param(jnp.ones(ffw_size))



  def __call__(self, x):
    moe_config = self.config.moe_config
    # x[B, T, D]
    # [D, 2F]
    ffw_size = self.config.ffw_size
    t = self.pre_norm(x)

    """
    MoE:
    one router for all 3 proj matrices in matrix
    router: [D, E], softmax and do weighted sum
    glu: [E, D, 2F]
    down_proj: [E, F, D]

    get top k experts for each token:
    [B, T, k]

    flatten router experts into [B*T, k]

    create inds of size [B*T, k] (repeated over k axis)
    flatten into [B*T*k]
    flatten router experts into [B*T*k]
    sort router experts into [B*T*k]
    create group sizes
    create flattened sortened input of [B*T*k, D]
    use ragged dot
    unsort input into [B*T*k, D]
    reshape to [B, T, k, D]
    multiply by expert weights and sum

    flatten input to [B*T, k]

    """


    if moe_config:
      k, E = moe_config.experts_per_token, moe_config.experts
      # [B, T, E]
      router_logits = jnp.einsum('btd,de->bte', t, self.router)
      # [B, T, E]
      router_weights = jax.nn.softmax(router_logits, axis=-1)
      # [B, T, k] [B, T, k]
      top_k_weights, top_k_experts = jax.lax.top_k(router_weights, k)

      B, T, D = t.shape
      # [B * T, D]
      flattened_x = t.reshape((B*T, D))
      # [B * T, k]
      flattened_x_inds_padded = jnp.repeat(jnp.arange(B*T)[:, None], k, axis=-1)
      # [B*T*k]
      flattened_x_inds_padded = flattened_x_inds_padded.reshape((B*T*k,))
      # [B*T*k]
      flattened_routed_experts = top_k_experts.reshape((B*T*k,))
      # [B*T*k]
      sorted_routed_experts_inds = jnp.argsort(flattened_routed_experts)
      # [B*T*k]
      # sorted_routed_experts = flattened_routed_experts[sorted_routed_experts_inds]
      # [E]
      sizes = jnp.bincount(flattened_routed_experts, minlength=E, length=E)


      # [B*T*k]
      sorted_flattened_x_inds = flattened_x_inds_padded[sorted_routed_experts_inds]
      # [B*T*k, D]
      sorted_flat_x = flattened_x[sorted_flattened_x_inds]
      # x[B*T*k, D]
      # W[E, D, 2F]
      # => [B*T*k, 2F]
      glu = jax.lax.ragged_dot(sorted_flat_x, self.glu_proj.value, sizes)
      # [B*T*k, F]
      f1 = jax.lax.dynamic_slice_in_dim(glu, 0, ffw_size, axis=-1)
      # [B*T*k, F]
      f2 = jax.lax.dynamic_slice_in_dim(glu, ffw_size, ffw_size, axis=-1)
      # [B*T*k, F]
      swiglu_out = f1 * f2 * jax.nn.sigmoid(self.beta * f2)
      # [B*T*k, D]
      out = jax.lax.ragged_dot(swiglu_out, self.down_proj.value, sizes)
      # [B*T*k, D]
      out = out[jnp.argsort(sorted_flattened_x_inds)]
      # [B, T, k, D]
      out = out.reshape((B, T, k, D))
      # w[B, T, k] -> [B, T, k, 1]
      # out[B, T, k, D]
      # => [B, T, k, D] => [B, T, D]
      out = jnp.sum(top_k_weights[..., None] * out, axis=2)
    else:
      glu = jnp.einsum('btd,df->btf', t, self.glu_proj)
      # [B, T, F]
      f1 = jax.lax.dynamic_slice_in_dim(glu, 0, ffw_size, axis=-1)
      # [B, T, F]
      f2 = jax.lax.dynamic_slice_in_dim(glu, ffw_size, ffw_size, axis=-1)
      t = f1 * f2 * jax.nn.sigmoid(self.beta * f2)
      out = jnp.einsum('btf,fd->btd', t, self.down_proj)

    return x + out


# TEST
@dataclass
class MoEConfig:
  experts_per_token: int
  experts: int

@dataclass
class ModelConfig:
  hidden_size: int
  ffw_size: int
  layers: int
  attn_heads: int
  vocab_size: int
  moe_config: MoEConfig | None


B, T, D, F, k, E = 2, 2, 4, 16, 2, 4

moe_config = MoEConfig(
    experts_per_token=k,
    experts=E
)

config = ModelConfig(
    hidden_size=D,
    ffw_size=F,
    layers=2,
    attn_heads=2,
    vocab_size=3,
    moe_config=moe_config
)

rngs = nnx.Rngs(0)
moe_mlp = MLP(config, rngs=rngs)
x = jax.random.normal(rngs.params(), (B, T, D))
print('x shape:', x.shape)
moe_out = moe_mlp(x)
print('moe_out shape:', moe_out.shape)

# END TEST


class TransformerLayer(nnx.Module):
  def __init__(self, config, *, rngs):
    self.attn_block = MHA(config, rngs=rngs)
    self.mlp_block = MLP(config, rngs=rngs)

  def __call__(self, x):
    t = self.attn_block(x)
    return self.mlp_block(t)


class Transformer(nnx.Module):
  def __init__(self, config, *, rngs):
    hidden_size = config.hidden_size
    vocab_size = config.vocab_size
    self.embed = nnx.Param(0.02 * jax.random.normal(rngs.params(), (vocab_size, hidden_size)))
    self.layers = [TransformerLayer(config, rngs=rngs) for _ in config.layers]

  def __call__(self, x):
    x = self.embed[x]
    for l in self.layers:
      x = l(x)
    return jnp.einsum('btd,vd->btv', x, self.embed)


x shape: (2, 2, 4)
moe_out shape: (2, 2, 4)


basic training loop

In [3]:
import jax
import jax.numpy as jnp

"""

x[B, T, D]

MoE:
one router for all 3 proj matrices in matrix
router: [D, E], softmax and do weighted sum
glu: [E, D, 2F]
down_proj: [E, F, D]

get top k experts for each token:
[B, T, k]

flatten router experts into [B*T, k]

create inds of size [B*T, k] (repeated over k axis)
flatten into [B*T*k]
flatten router experts into [B*T*k]
sort router experts into [B*T*k]
create group sizes
create flattened sortened input of [B*T*k, D]
use ragged dot
unsort input into [B*T*k, D]
reshape to [B, T, k, D]
multiply by expert weights and sum

flatten input to [B*T, k]

"""
B, T, D = 1, 3, 4
rngs = nnx.Rngs(0)
ffw_size = 4*D
x = jax.random.normal(rngs.params(), (B, T, D))
hidden_size = D
k = 2

E = 4

router = 0.02 * jax.random.normal(rngs.params(), (hidden_size, E))
glu_proj = 0.02 * jax.random.normal(rngs.params(), (E, hidden_size, 2*ffw_size))
down_proj = 0.02 * jax.random.normal(rngs.params(), (E, ffw_size, hidden_size))
beta = jnp.ones(ffw_size)

"""
logic
"""
# [B, T, E]
router_logits = jnp.einsum('btd,de->bte', x, router)
# [B, T, E]
router_weights = jax.nn.softmax(router_logits, axis=-1)
# [B, T, k] [B, T, k]
top_k_weights, top_k_experts = jax.lax.top_k(router_weights, k)

B, T, D = x.shape
# [B * T, D]
flattened_x = x.reshape((B*T, D))
# [B * T, k]
flattened_x_inds_padded = jnp.repeat(jnp.arange(B*T)[:, None], k, axis=-1)
# [B*T*k]
flattened_x_inds_padded = flattened_x_inds_padded.reshape((B*T*k,))
# [B*T*k]
flattened_routed_experts = top_k_experts.reshape((B*T*k,))
# [B*T*k]
sorted_routed_experts_inds = jnp.argsort(flattened_routed_experts)
# [B*T*k]
sorted_routed_experts = flattened_routed_experts[sorted_routed_experts_inds]
# [B*T*k]
sorted_flattened_x_inds = flattened_x_inds_padded[sorted_routed_experts_inds]
# [E]
sizes = jnp.bincount(sorted_routed_experts, minlength=E, length=E)
# [B*T*k, D]
sorted_flat_x = flattened_x[sorted_flattened_x_inds]
# x[B*T*k, D]
# W[E, D, 2F]
# => [B*T*k, 2F]
glu = jax.lax.ragged_dot(sorted_flat_x, glu_proj, sizes)
# [B*T*k, F]
f1 = jax.lax.dynamic_slice_in_dim(glu, 0, ffw_size, axis=-1)
# [B*T*k, F]
f2 = jax.lax.dynamic_slice_in_dim(glu, ffw_size, ffw_size, axis=-1)
# [B*T*k, F]
swiglu_out = f1 * f2 * jax.nn.sigmoid(beta * f2)
# [B*T*k, D]
out = jax.lax.ragged_dot(swiglu_out, down_proj, sizes)
# [B*T*k, D]
out = out[jnp.argsort(sorted_flattened_x_inds)]
# [B, T, k, D]
out = out.reshape((B, T, k, D))
# w[B, T, k] -> [B, T, k, 1]
# out[B, T, k, D]
# => [B, T, k, D] => [B, T, D]
print('B:', B, 'T:', T, 'k:', k, 'D:', D)
out = jnp.sum(top_k_weights[..., None] * out, axis=2)
print(out.shape)


NameError: name 'nnx' is not defined